In [ ]:
#| default_exp git

# git

> Did the change land, and if not, what happened to it.

An agent that edited a file an hour ago has no way to find out what became of the edit. It might
be committed. It might be sitting in the working tree. Somebody might have written over it. The
three are worth telling apart before planning the next move, and none of them is visible from
the transcript.

`landed` tells them apart from the record. A touch stored the hashes of the lines it added;
`git blame` says which commit owns each line of the file now. Matching one against the other
gives a count of how many of those lines survived and who owns them.

In [ ]:
#| export
from dataclasses import dataclass, field
from pathlib import Path

from fastcore.basics import AttrDict, patch
from fastcore.foundation import L
from gheasy.repo import GitError, GitRepo

from panjika.core import DETAIL, Home, file_hash, fold, git_root, hashed, now
from panjika.read import Ledger, _since

`STATES` names what became of the lines a session wrote. `EVIDENCE` says how sure the answer is: `lines` matched the exact lines against `git blame`, `path` only knows that some commit touched the file afterwards, and `none` had neither to go on.

In [ ]:
#| export
STATES = ('landed', 'partly_landed', 'pending', 'replaced', 'gone', 'untracked',
          'uncertain', 'unknown')
EVIDENCE = ('lines', 'path', 'none')

UNCOMMITTED = '0' * 40

## A verdict

In [ ]:
#| export
@dataclass
class Verdict:
    "What became of one session's change to one file."
    path: str
    state: str = 'unknown'
    why: str = ''
    kept: int = 0
    total: int = 0
    evidence: str = 'none'
    session: str = ''
    action: str = 'edit'
    commits: list = field(default_factory=list)
    branch: str = ''
    branch_gone: bool = False
    elsewhere: list = field(default_factory=list)

    @property
    def done(self):
        "Whether the change is in the history and needs nothing further."
        return self.state == 'landed'

    @property
    def anywhere(self):
        "Whether the lines are still somewhere in this repository."
        return self.state in ('landed', 'partly_landed', 'pending') or bool(self.elsewhere)

    @property
    def survived(self):
        "The share of the lines this session wrote that are still in the file."
        return 0.0 if not self.total else round(self.kept / self.total, 3)

    def line(self):
        "One line, for a terminal or a tool result."
        share = '' if not self.total else f'  {self.kept}/{self.total} lines'
        return f'{self.state:<14} {self.path}{share}  {self.why}'.rstrip()

    def dict(self):
        "The verdict as a record, with the computed fields spelled out."
        return AttrDict(self.__dict__ | {'survived': self.survived, 'anywhere': self.anywhere})

## Reading the file back

`blame_rows` is the only place git is asked anything. An untracked file has no blame, and a
missing one has no file, so both answer with a state rather than an exception.

`match_lines` pairs each recorded hash with an unclaimed line of the file that hashes the same. Matching on content rather than on line number survives the file being reformatted, moved around, or added to above the change.

In [ ]:
#| export
def blame_rows(repo, path):
    "Every line of `path` with the commit that owns it, or `None` when git will not blame it."
    try: return repo.blame(str(path))
    except (GitError, Exception): return None


def commit_row(repo, oid, cache=None):
    "One commit as the few fields a verdict quotes."
    if cache is not None and oid in cache: return cache[oid]
    try:
        d = repo.commit_detail(oid)
        row = AttrDict(sha=d['oid'], short=d['short'], subject=d['message'].splitlines()[0][:120],
                       author=d['author'], at=d['timestamp'])
    except Exception:
        row = AttrDict(sha=oid, short=oid[:7], subject='', author='', at=0)
    if cache is not None: cache[oid] = row
    return row


def match_lines(wanted, rows):
    "Pair each recorded line hash with an unclaimed line of the file that hashes the same."
    pool = {}
    for r in rows: pool.setdefault(hashed(r.get('text', '')), []).append(r)
    found, missing = [], 0
    for h in wanted:
        bucket = pool.get(h)
        if bucket: found.append(bucket.pop(0))
        else: missing += 1
    return found, missing

## Branches

A verdict is about one working tree. The session recorded which branch it ran on, and that branch may since have been merged and deleted, or may not be the one checked out now. `landed` answers for the current checkout, so a change made on a feature branch reads as `replaced` from `main` and reads correctly from the branch itself.

`surviving_branches` is the second question. A change absent from this branch may still be on another, and a change on no branch at all is gone from the repository rather than merely from here. Both are answered by hashing each branch's copy of the file and matching the lines the session recorded, which needs no blame and no checkout.

In [ ]:
#| export
def local_branches(repo):
    "Every local branch name."
    try: out = repo.run('for-each-ref', '--format=%(refname:short)', 'refs/heads/')
    except Exception: return []
    return [b.strip() for b in out.splitlines() if b.strip()]


def lines_on(repo, branch, path, wanted):
    "How many of the recorded line hashes `path` still holds on `branch`."
    try: text = repo.run('show', f'{branch}:{path}')
    except Exception: return 0
    pool = {}
    for line in text.splitlines():
        h = hashed(line)
        pool[h] = pool.get(h, 0) + 1
    kept = 0
    for h in wanted:
        if pool.get(h): pool[h] -= 1; kept += 1
    return kept


def surviving_branches(repo, path, wanted, skip=''):
    "The branches, other than `skip`, holding every line the session wrote."
    if not wanted: return []
    return [b for b in local_branches(repo)
            if b != skip and lines_on(repo, b, path, wanted) == len(wanted)]


def _head_branch(repo):
    "The branch checked out now, or `''` on a detached HEAD."
    try: name = repo.run('rev-parse', '--abbrev-ref', 'HEAD').strip()
    except Exception: return ''
    return '' if name == 'HEAD' else name


def _wanted(ledger, touches):
    "Every line hash the session recorded for one path, oldest touch first."
    return [h for t in touches for h in ((ledger.detail(t.id) or {}).get('lines') or ())]

## The verdict

`_coarse` answers when the machine-local tier is not here to say which lines were written. A later commit touching the file is as consistent with the change being reverted as with it surviving, so the verdict is `uncertain` rather than `landed`. Calling it `landed` would tell an agent to skip work it still has to do.

`history` in gheasy defaults to `--all`, which reaches commits on other branches. `_replaced_by` asks for `HEAD`, because a commit that is not in this branch's history did not replace anything here.

In [ ]:
#| export
def _replaced_by(repo, path, after):
    "The commits on this branch that touched `path` after a session did, newest first."
    try: rows = repo.history(limit=20, ref='HEAD', path=str(path))
    except Exception: return []
    return [r for r in rows if (r.get('timestamp') or 0) >= after - 1]


def _coarse(repo, touch, root, why_extra='', after=None):
    "The verdict when the machine-local tier is not here to say which lines were written."
    path = touch.path
    after = touch.get('at') or 0 if after is None else after
    later = _replaced_by(repo, path, after)
    v = Verdict(path=path, session=touch.session, action=touch.get('action', 'edit'),
                evidence='path', commits=[AttrDict(sha=c['oid'], short=c['short'],
                                                   subject=c['subject'], author=c['author'],
                                                   at=c['timestamp']) for c in later])
    if later:
        v.state = 'uncertain'
        v.why = (f'{len(later)} commit(s) touched this file afterwards, but no line record is '
                 'here to say whether this change is still in it. Read the commits, or run '
                 'where the session ran.')
        return v
    if file_hash(root/path) == touch.get('after'):
        v.state, v.why = 'pending', 'the file is byte-for-byte what the session left, and nothing has been committed'
        return v
    v.state = 'unknown'
    v.why = ('no line record and no commit since. ' + why_extra).strip()
    return v

`touches` is every touch the session made to that path, oldest first. A session that commits halfway and then edits again has written lines in two places, some in the history and some still in the working tree, so all of them are read rather than only the last.

A merged branch is deleted, which is the ordinary end of one. `branch_gone` is a field on every verdict, but it only reaches `why` when the state is not `landed`, because on a change that landed the branch being gone is not a concern.

In [ ]:
#| export
def verdict_for(ledger,
                touches,        # every touch that session made to one path
                repo=None,      # the repository, built from `root` when absent
                root=None,      # the repository root, taken from `repo` when absent
                cache=None,     # a commit-row cache shared across one `landed` call
                branch=''):     # the branch the session ran on
    "What became of a session's whole change to one file."
    if not isinstance(touches, (list, tuple, L)): touches = [touches]
    touches = sorted(touches, key=lambda t: t.get('at') or 0)
    touch, branch = touches[-1], str(branch or '')
    root = Path(root or repo.root)
    repo = repo or GitRepo(root)
    path, action = touch.path, touch.get('action', 'edit')
    target, after = root/path, touches[0].get('at') or 0
    wanted, here = _wanted(ledger, touches), _head_branch(repo)
    gone = bool(branch) and branch != here and branch not in local_branches(repo)

    def out(v):
        v.branch, v.branch_gone = branch, gone
        if v.state in ('replaced', 'gone') and v.total:
            v.elsewhere = surviving_branches(repo, path, wanted, here)
            if v.elsewhere: v.why += f'; still on {", ".join(v.elsewhere)}'
            else: v.why += '; and on no other branch either'
        if gone and v.state != 'landed':
            v.why = f'{v.why}; the branch {branch} it ran on is gone'.lstrip('; ')
        return v

    if not target.exists():
        gone_in = _replaced_by(repo, path, after)
        if action == 'delete' and gone_in:
            return out(Verdict(path, 'landed', 'the deletion is in the history', evidence='path',
                               session=touch.session, action=action,
                               commits=[commit_row(repo, c['oid'], cache) for c in gone_in[:1]]))
        return out(Verdict(path, 'gone', 'the file is not in the working tree', evidence='none',
                           session=touch.session, action=action, total=len(wanted)))
    if not touch.get('tracked', True) and not _replaced_by(repo, path, after):
        return out(Verdict(path, 'untracked', 'git is not tracking this file, so nothing can land',
                           evidence='none', session=touch.session, action=action))
    rows = blame_rows(repo, path)
    if rows is None: return out(_coarse(repo, touch, root, 'git would not blame this path', after))
    if not wanted: return out(_coarse(repo, touch, root, 'the session recorded no added lines', after))

    found, missing = match_lines(wanted, rows)
    kept, total = len(found), len(wanted)
    oids = [r['oid'] for r in found]
    committed = sorted({o for o in oids if o != UNCOMMITTED})
    waiting = sum(1 for o in oids if o == UNCOMMITTED)
    commits = [commit_row(repo, o, cache) for o in committed]
    v = Verdict(path=path, session=touch.session, action=action, kept=kept, total=total,
                evidence='lines', commits=commits)
    if kept == 0:
        later = _replaced_by(repo, path, after)
        who = f'; {later[0]["short"]} {later[0]["subject"][:60]!r} by {later[0]["author"]} owns it now' if later else ''
        v.state, v.why = 'replaced', f'none of the {total} lines this session wrote are in the file{who}'
        v.commits = [commit_row(repo, later[0]['oid'], cache)] if later else []
    elif not committed:
        v.state, v.why = 'pending', f'all {kept} surviving lines are uncommitted in the working tree'
    elif kept < total:
        v.state = 'partly_landed'
        v.why = f'{total - kept} of {total} lines are gone; the rest are in ' \
                f'{", ".join(c.short for c in commits)}'
    elif waiting:
        v.state = 'partly_landed'
        v.why = f'{waiting} of {total} lines are still uncommitted; the rest are in ' \
                f'{", ".join(c.short for c in commits)}'
    else:
        v.state = 'landed'
        v.why = f'all {total} lines are in {", ".join(c.short for c in commits)}'
    return out(v)

## The call an agent makes

`landed` is the whole planning surface. Given nothing it reads the newest session; given a
session id it reads that one; given a path it reads every session that touched it.

In [ ]:
#| export
def landed(session=None, path='', home=None, start='.', ledger=None):
    "What became of what was written, as one `Verdict` per file. Answers for the newest session by default."
    led = ledger or Ledger(home, start)
    root = git_root(led.home.root) or git_root(start)
    if root is None: return L()
    repo, cache = GitRepo(root), {}
    on = {r.session: r.get('branch') or '' for r in fold(led.of('session'))}
    if path and not session:
        groups = [[t for t in led.of('touch') if t.session == r.session and t.get('path') == path]
                  for r in led.trail(path)]
    else:
        if session in (None, '', 'latest'):
            rows = led.sessions(limit=1)
            if not rows: return L()
            session = rows[0].session
        by_path = {}
        for t in led.of('touch'):
            if t.session != session: continue
            if path and t.get('path') != path: continue
            by_path.setdefault(t.path, []).append(t)
        groups = list(by_path.values())
    return L(verdict_for(led, g, repo, root, cache, on.get(g[0].session, '')) for g in groups if g)


def report(session=None, path='', home=None, start='.', ledger=None):
    "The same answers as one row a model can read, plus the counts."
    vs = landed(session, path, home, start, ledger)
    counts = {}
    for v in vs: counts[v.state] = counts.get(v.state, 0) + 1
    order = ' · '.join(f'{n} {s}' for s, n in sorted(counts.items(), key=lambda kv: -kv[1]))
    return AttrDict(verdicts=vs, counts=counts, summary=order or 'nothing recorded',
                    text='\n'.join(v.line() for v in vs) or 'nothing recorded')

## Linking a commit to the sessions that earned it

A commit rarely names the session that wrote it, because the person committing is not the agent
that edited. `link_commit` closes that gap from the other end: it takes a commit, looks at the
files in it, and records it against every session that touched one of them recently. That is
what the `post-commit` hook calls, so the link is made whoever ran `git commit`.

In [ ]:
#| export
def link_commit(sha='HEAD', home=None, start='.', lookback='7d'):
    "Record `sha` against every session that recently touched a file in it. Returns those sessions."
    from panjika.write import Scribe
    led = Ledger(home, start)
    root = git_root(led.home.root) or git_root(start)
    if root is None: return L()
    repo = GitRepo(root)
    try: d = repo.commit_detail(sha)
    except Exception: return L()
    files = [ln.split()[0] for ln in repo.run('show', '--name-only', '--format=', d['oid']).splitlines() if ln.strip()]
    if not files: return L()
    after, seen = _since(lookback), {}
    for r in led.of('touch'):
        if r.get('path') in files and (r.get('at') or 0) >= after: seen[r.session] = r.at
    already = {(c.session, c.get('sha')) for c in led.of('commit')}
    out = L()
    for sid in seen:
        if (sid, d['oid']) in already: continue
        Scribe(home=led.home, session=sid, start=start, detail=False).commit(
            d['oid'], subject=d['message'].splitlines()[0], author=d['author'],
            files=files, branch=repo.run('rev-parse', '--abbrev-ref', 'HEAD').strip())
        out.append(sid)
    return out

## One file, both histories

`blend` is what a history view draws: the commits of a file and the agent sessions that touched
it, on one timeline. A commit that no session earned is still a commit, and a session that never
landed is still on the list.

git stamps a commit to the second while a touch is stamped to the millisecond, so `blend` compares the two at the coarser of them. Within one second the commit is the later fact, because it carries work that had to exist before it.

In [ ]:
#| export
def blend(path, home=None, start='.', ledger=None, limit=60):
    "The commits and the agent sessions for one file, newest first, on one list."
    led = ledger or Ledger(home, start)
    root = git_root(led.home.root) or git_root(start)
    out = L()
    for row in led.trail(path, limit=limit):
        out.append(AttrDict(kind='session', at=row.at, session=row.session,
                            harness=row.get('harness', ''), model=row.get('model', ''),
                            branch=row.get('branch', ''),
                            title=row.get('title') or row.get('prompt', ''),
                            added=row.touch.get('added', 0), removed=row.touch.get('removed', 0)))
    if root is not None:
        try: rows = GitRepo(root).history(limit=limit, path=str(path))
        except Exception: rows = []
        for c in rows:
            out.append(AttrDict(kind='commit', at=c['timestamp'], sha=c['oid'], short=c['short'],
                                title=c['subject'], author=c['author']))
    return out.sorted(key=lambda r: (int(r.at), r.kind == 'commit'), reverse=True)[:int(limit)]

## Trying it

Nothing is committed yet, so the change is waiting in the working tree.

In [ ]:
import subprocess, tempfile
from fastcore.test import test_eq
from panjika.write import Scribe

def _git(root, *a): subprocess.run(['git', *a], cwd=root, capture_output=True, check=True)

d = Path(tempfile.mkdtemp())/'proj'; d.mkdir(parents=True)
_git(d, 'init', '-q', '-b', 'main')
_git(d, 'config', 'user.email', 'a@b.c'); _git(d, 'config', 'user.name', 'T')
(d/'app.py').write_text('def add(a, b):\n    return a + b\n')
_git(d, 'add', '-A'); _git(d, 'commit', '-qm', 'first')

sc = Scribe(home=d/'.panjika', start=d); sc.home.init()
sc.begin('claude-code', model='opus-5', prompt='handle strings')
(d/'app.py').write_text('def add(a, b):\n    if isinstance(a, str): return a + str(b)\n    return a + b\n')
sc.touch(d/'app.py', 'edit', sc.step('Edit', target='app.py'))
sc.end('done')

led = Ledger(d/'.panjika')
v = landed(sc.session, ledger=led)[0]
test_eq((v.state, v.kept, v.total, v.evidence), ('pending', 1, 1, 'lines'))

Commit it, and the same question names the commit that carries it.

In [ ]:
_git(d, 'commit', '-aqm', 'handle strings')
led = Ledger(d/'.panjika')
v = landed(sc.session, ledger=led)[0]
test_eq((v.state, v.kept, v.total), ('landed', 1, 1))
test_eq(v.commits[0].subject, 'handle strings')
assert v.done

The session writes more after part of it landed, so its work is in two places at once.

In [ ]:
(d/'app.py').write_text('def add(a, b):\n    if isinstance(a, str): return a + str(b)\n'
                        '    if b is None: return a\n    return a + b\n')
sc.touch(d/'app.py', 'edit', sc.step('Edit', target='app.py'))
v = landed(sc.session, ledger=Ledger(d/'.panjika'))[0]
test_eq((v.state, v.kept, v.total), ('partly_landed', 2, 2))
assert 'still uncommitted' in v.why

Somebody writes over it. The verdict says so and names them, and the commit that owns the lines now is a field rather than only a phrase inside `why`. One commit, not two: the commit that landed the change also touched this file after the session began, and naming it here would say it replaced itself.

In [ ]:
(d/'app.py').write_text('def add(a, b):\n    return a + b\n')
_git(d, 'commit', '-aqm', 'revert that')
led = Ledger(d/'.panjika')
v = landed(sc.session, ledger=led)[0]
test_eq((v.state, v.kept), ('replaced', 0))
assert 'revert that' in v.why
test_eq([c.subject for c in v.commits], ['revert that'])
assert v.commits[0].sha and v.commits[0].author

The link is made from the commit end, whoever ran `git commit`.

In [ ]:
led = Ledger(d/'.panjika')
test_eq(link_commit('HEAD', home=d/'.panjika', start=d), [sc.session])
test_eq(Ledger(d/'.panjika').session(sc.session).commits[0].subject, 'revert that')

The post-commit hook fires on every commit, so linking the same one twice records it once, and a commit touching nothing a session touched links to nobody.

In [ ]:
test_eq(link_commit('HEAD', home=d/'.panjika', start=d), [])
test_eq(len(Ledger(d/'.panjika').session(sc.session).commits), 1)

(d/'unrelated.py').write_text('Z = 1\n')
_git(d, 'add', 'unrelated.py'); _git(d, 'commit', '-qm', 'nothing to do with any session')
test_eq(link_commit('HEAD', home=d/'.panjika', start=d), [])

One file, both histories, on one timeline.

In [ ]:
rows = blend('app.py', home=d/'.panjika', start=d)
test_eq([r.kind for r in rows][:1], ['commit'])
test_eq(sorted({r.kind for r in rows}), ['commit', 'session'])

The two states that are not about lines at all. A file git was never told about cannot land however good the change is, and a file that is no longer there is not a verdict about lines.

In [ ]:
sc2 = Scribe(home=d/'.panjika', start=d)
sc2.begin('codex', model='gpt-5.6', prompt='add a helper')
(d/'helper.py').write_text('def half(x): return x / 2\n')
sc2.touch(d/'helper.py', 'create', sc2.step('Write', target='helper.py'))
test_eq(landed(sc2.session, ledger=Ledger(d/'.panjika'))[0].state, 'untracked')

(d/'helper.py').unlink()
test_eq(landed(sc2.session, ledger=Ledger(d/'.panjika'))[0].state, 'gone')

A session writes two lines, both land, and somebody takes one out again. This is the other route to `partly_landed`. Reported as `landed` it would tell an agent a change is done when half of it is gone.

In [ ]:
sc3 = Scribe(home=d/'.panjika', start=d)
sc3.begin('claude-code', model='opus-5', prompt='two guards')
(d/'app.py').write_text('def add(a, b):\n    first = 1\n    second = 2\n    return a + b\n')
sc3.touch(d/'app.py', 'edit', sc3.step('Edit', target='app.py'))
sc3.end('done')
_git(d, 'commit', '-aqm', 'both guards')

(d/'app.py').write_text('def add(a, b):\n    first = 1\n    return a + b\n')
_git(d, 'commit', '-aqm', 'drop the second one')
v = landed(sc3.session, ledger=Ledger(d/'.panjika'))[0]
test_eq((v.state, v.kept, v.total), ('partly_landed', 1, 2))
assert '1 of 2 lines are gone' in v.why

The two calls the documentation leads with. `landed()` answers for the newest session, which is what an agent resuming its own work asks first. `landed(path=...)` answers for every session that touched one file, which is what it asks before editing.

In [ ]:
test_eq([v.session for v in landed(home=d/'.panjika', start=d)], [sc3.session])
test_eq({v.session for v in landed(path='app.py', home=d/'.panjika', start=d)},
        {sc.session, sc3.session})
test_eq(landed(path='helper.py', home=d/'.panjika', start=d)[0].state, 'gone')

On the far side of a clone the committed ledger travels and the machine-local line record does not, so the answer degrades to `uncertain` and says why.

In [ ]:
for p in (d/'.panjika'/'detail').glob('*.jsonl'): p.unlink()
v = landed(sc.session, ledger=Ledger(d/'.panjika'))[0]
test_eq((v.state, v.evidence), ('uncertain', 'path'))
assert 'no line record' in v.why

## Which branch, and is it still there

A session works on a branch. Read from another branch the same change reads as `replaced`, and read after that branch is merged and deleted it reads as `replaced` with nothing to say why. The verdict carries the branch the session ran on, says when that branch is gone, and names the branches that still hold the lines.

In [ ]:
b = Path(tempfile.mkdtemp())/'proj'; b.mkdir(parents=True)
_git(b, 'init', '-q', '-b', 'main')
_git(b, 'config', 'user.email', 'a@b.c'); _git(b, 'config', 'user.name', 'Sam')
(b/'app.py').write_text('def add(a, b):\n    return a + b\n')
_git(b, 'add', '-A'); _git(b, 'commit', '-qm', 'first')

_git(b, 'checkout', '-q', '-b', 'feature')
bs = Scribe(home=b/'.panjika', start=b); bs.home.init()
bs.begin('claude-code', model='opus-5', prompt='handle strings')
(b/'app.py').write_text('def add(a, b):\n    if isinstance(a, str): return a + str(b)\n    return a + b\n')
bs.touch(b/'app.py', 'edit', bs.step('Edit', target='app.py'))
bs.end('done')
_git(b, 'commit', '-aqm', 'handle strings')

v = landed(bs.session, ledger=Ledger(b/'.panjika'))[0]
test_eq((v.state, v.branch, v.branch_gone), ('landed', 'feature', False))
assert v.anywhere

From `main` the change is not here. The commit named as owning those lines is the one on this branch, never the feature commit that carries the change, because `_replaced_by` asks for `HEAD` rather than every ref. `elsewhere` names where the change actually went.

In [ ]:
_git(b, 'checkout', '-q', 'main')
v = landed(bs.session, ledger=Ledger(b/'.panjika'))[0]
test_eq((v.state, v.branch, v.elsewhere), ('replaced', 'feature', ['feature']))
assert v.anywhere
assert 'still on feature' in v.why
test_eq([c.subject for c in v.commits], ['first'])

In [ ]:
_git(b, 'branch', '-qD', 'feature')
v = landed(bs.session, ledger=Ledger(b/'.panjika'))[0]
test_eq((v.state, v.elsewhere, v.branch_gone), ('replaced', [], True))
assert not v.anywhere
assert 'no other branch' in v.why and 'the branch feature it ran on is gone' in v.why

A branch that was merged and then deleted is the ordinary end of a branch, so the verdict stays `landed` and says nothing alarming.

In [ ]:
_git(b, 'checkout', '-q', '-b', 'merged-work')
ms = Scribe(home=b/'.panjika', start=b)
ms.begin('claude-code', model='opus-5', prompt='add a guard')
(b/'app.py').write_text('def add(a, b):\n    if a is None: return b\n    return a + b\n')
ms.touch(b/'app.py', 'edit', ms.step('Edit', target='app.py'))
ms.end('done')
_git(b, 'commit', '-aqm', 'add a guard')
_git(b, 'checkout', '-q', 'main')
_git(b, 'merge', '-q', '--no-ff', '--no-edit', 'merged-work')
_git(b, 'branch', '-qd', 'merged-work')

v = landed(ms.session, ledger=Ledger(b/'.panjika'))[0]
test_eq((v.state, v.branch_gone), ('landed', True))
assert v.anywhere and 'is gone' not in v.why

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()